# PennyLane fidelity quantum kernel

Construct a feature-map kernel from QNode state overlaps on default.qubit and MettleQ.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
data = np.asarray([[0.1, 0.2], [0.7, -0.4], [-0.5, 0.6], [0.2, 0.9]])

def make_state_qnode(device):
    @qml.qnode(device)
    def circuit(values):
        for wire in range(2):
            qml.Hadamard(wire)
            qml.RZ(values[wire], wires=wire)
        qml.IsingZZ(values[0] * values[1], wires=[0, 1])
        return qml.state()
    return circuit

def kernel(qnode):
    states = [np.asarray(qnode(row)) for row in data]
    return np.asarray([[abs(np.vdot(left, right)) ** 2 for right in states] for left in states])

reference_qnode = make_state_qnode(qml.device("default.qubit", wires=2))
reference, reference_ms, _ = benchmark(lambda: kernel(reference_qnode))
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_state_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: kernel(mettleq_qnode))
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/09_quantum_kernel.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="kernel matrix atol=4e-6",
    passed=error <= 4e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_kernel_error": error, "reference": reference, "mettleq": candidate},
)